In [1]:
"""
Advanced Time Series to Images Transformation

Transformation avancée des séries temporelles en images avec trois méthodes:
- GASF (Gramian Angular Summation Field)
- MTF (Markov Transition Field)
- RP (Recurrence Plot)

Améliorations:
- Fusion multi-canal pour capteurs SeizeIT2
- Optimisation des paramètres de transformation
- Normalisation adaptative
- Support images RGB multi-canal
"""

'\nAdvanced Time Series to Images Transformation\n\nTransformation avancée des séries temporelles en images avec trois méthodes:\n- GASF (Gramian Angular Summation Field)\n- MTF (Markov Transition Field)\n- RP (Recurrence Plot)\n\nAméliorations:\n- Fusion multi-canal pour capteurs SeizeIT2\n- Optimisation des paramètres de transformation\n- Normalisation adaptative\n- Support images RGB multi-canal\n'

# B. Advanced Time Series to Images Transformation

Transformation avancée des séries temporelles en images avec trois méthodes:
- **GASF** (Gramian Angular Summation Field)
- **MTF** (Markov Transition Field)
- **RP** (Recurrence Plot)

**Améliorations**:
- Fusion multi-canal pour capteurs SeizeIT2
- Optimisation des paramètres de transformation
- Normalisation adaptative
- Support images RGB multi-canal

In [2]:
# Install required packages
!pip install pyts scikit-image scipy matplotlib seaborn tqdm google.colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.7 MB/s eta 0:00:00


In [3]:
import numpy as np
import gc
import matplotlib.pyplot as plt
from pyts.image import GramianAngularField, MarkovTransitionField, RecurrencePlot
from sklearn.preprocessing import MinMaxScaler
from skimage.transform import resize
from scipy import signal
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import os
import seaborn as sns
# First handle the drive mounting properly
from google.colab import drive
drive.flush_and_unmount()  # Unmount if mounted
!rm -rf /content/drive  # Clear any remaining files
drive.mount('/content/drive')  # Fresh mount

# Configuration des chemins
BASE_PATH = '/content/drive/MyDrive/SeizeIT2_Project/'
DATA_SPLIT_PATH = BASE_PATH + 'data_split/'
IMAGE_DATA_PATH = BASE_PATH + 'image_data/'
PLOTS_PATH = BASE_PATH + 'plots/'

# Créer les dossiers pour sauvegarder les images
for method in ['gasf', 'mtf', 'rp']:
    os.makedirs(IMAGE_DATA_PATH + method + '/', exist_ok=True)

print("🔧 Configuration terminée!")

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
🔧 Configuration terminée!


In [4]:
class OptimizedImageTransformer:
    """
    Transformateur optimisé pour convertir les signaux multi-canaux en images
    OPTIMISÉ POUR ÉVITER LES PROBLÈMES DE MÉMOIRE
    """

    def __init__(self, image_size=64, n_channels_to_use=3, batch_size=32):
        """
        Paramètres OPTIMISÉS:
        - image_size: Réduit à 64 pour économiser la mémoire
        - n_channels_to_use: Réduit à 3 canaux
        - batch_size: Traitement par lots de 32
        """
        self.image_size = image_size
        self.n_channels_to_use = n_channels_to_use
        self.batch_size = batch_size

        # Initialiser les transformateurs avec paramètres optimisés
        self.gasf = GramianAngularField(
            image_size=image_size,
            method='summation',
            sample_range=(-1, 1)
        )

        self.mtf = MarkovTransitionField(
            image_size=image_size,
            n_bins=6,  # Réduit pour économiser la mémoire
            strategy='quantile'
        )

        self.rp = RecurrencePlot(
            dimension=2,  # Réduit pour économiser la mémoire
            time_delay=1,
            threshold='point',
            percentage=10
        )

        self.scaler = MinMaxScaler(feature_range=(-1, 1))

        print(f"🛠️ Transformateur optimisé initialisé:")
        print(f"   Taille image: {image_size}x{image_size}")
        print(f"   Canaux utilisés: {n_channels_to_use}")
        print(f"   Taille batch: {batch_size}")

    def preprocess_signal(self, signal):
        """
        Prétraitement LÉGER du signal pour économiser la mémoire
        """
        # Normalisation simple
        signal_norm = self.scaler.fit_transform(signal.reshape(-1, 1)).flatten()

        # Filtrage léger seulement
        try:
            signal_filtered = signal.medfilt(signal_norm, kernel_size=3)
            return signal_filtered
        except:
            return signal_norm

    def process_batch(self, X_batch, transformer_method, method_name):
        """
        Traite un lot de données avec gestion de mémoire
        """
        batch_images = []

        for i in range(len(X_batch)):
            try:
                # Utiliser seulement les premiers canaux
                channels_data = []

                for ch in range(min(self.n_channels_to_use, X_batch.shape[1])):
                    # Prétraitement du signal
                    signal_clean = self.preprocess_signal(X_batch[i, ch, :])

                    # Transformer selon la méthode
                    if method_name == 'gasf':
                        img = self.gasf.fit_transform(signal_clean.reshape(1, -1))[0]
                    elif method_name == 'mtf':
                        img = self.mtf.fit_transform(signal_clean.reshape(1, -1))[0]
                    elif method_name == 'rp':
                        img = self.rp.fit_transform(signal_clean.reshape(1, -1))[0]
                        img = resize(img, (self.image_size, self.image_size), anti_aliasing=False)

                    channels_data.append(img)

                # Si moins de 3 canaux, dupliquer le dernier
                while len(channels_data) < 3:
                    channels_data.append(channels_data[-1])

                # Combiner en image RGB
                rgb_image = np.stack(channels_data[:3], axis=-1)
                batch_images.append(rgb_image)

                # Libérer la mémoire
                del signal_clean, img, channels_data, rgb_image

            except Exception as e:
                print(f"❌ Erreur échantillon {i}: {e}")
                # Créer une image vide en cas d'erreur
                empty_img = np.zeros((self.image_size, self.image_size, 3))
                batch_images.append(empty_img)

        return np.array(batch_images)

    def transform_with_batches(self, X, method_name):
      """
      Transformation avec traitement par lots pour économiser la mémoire
      """
      print(f"🔄 Transformation {method_name.upper()} par lots...")

      n_samples = X.shape[0]
      n_batches = (n_samples + self.batch_size - 1) // self.batch_size

      all_images = []

      for batch_idx in tqdm(range(n_batches), desc=f"{method_name.upper()}"):
          start_idx = batch_idx * self.batch_size
          end_idx = min(start_idx + self.batch_size, n_samples)

          # Extraire le lot
          X_batch = X[start_idx:end_idx]

          # Traiter le lot
          batch_images = self.process_batch(X_batch, None, method_name)
          all_images.append(batch_images)

          # Libérer la mémoire (CORRECTION ICI)
          del X_batch, batch_images
          import gc
          gc.collect()  # Force la libération de mémoire

      # Combiner tous les lots
      final_images = np.concatenate(all_images, axis=0)

      # Libérer la mémoire des lots
      del all_images
      gc.collect()

      return final_images

    def save_images_optimized(self, images, method, split):
        """
        Sauvegarde optimisée des images
        """
        save_path = IMAGE_DATA_PATH + f'{method}/'
        filename = f'X_{split}_{method}.npy'

        # Convertir en float32 pour économiser l'espace
        images_optimized = images.astype(np.float32)

        np.save(save_path + filename, images_optimized)
        print(f"💾 Sauvegardé: {filename} - Shape: {images_optimized.shape}")

        # Libérer la mémoire
        del images_optimized
        gc.collect()

    def calculate_image_statistics(self, images, method):
        """
        Calcule les statistiques de manière optimisée
        """
        print(f"\n📊 Statistiques {method.upper()}:")
        print(f"   Shape: {images.shape}")
        print(f"   Min: {images.min():.4f}")
        print(f"   Max: {images.max():.4f}")
        print(f"   Mean: {images.mean():.4f}")
        print(f"   Std: {images.std():.4f}")






In [5]:

def load_data_safely():
    """
    Chargement sécurisé des données avec vérification
    """
    print("📂 Chargement des données préparées...")

    try:
        X_train = np.load(DATA_SPLIT_PATH + 'X_train_raw.npy')
        X_val = np.load(DATA_SPLIT_PATH + 'X_val_raw.npy')
        X_test = np.load(DATA_SPLIT_PATH + 'X_test_raw.npy')
        y_train = np.load(DATA_SPLIT_PATH + 'y_train.npy')
        y_val = np.load(DATA_SPLIT_PATH + 'y_val.npy')
        y_test = np.load(DATA_SPLIT_PATH + 'y_test.npy')

        print(f"✅ Données chargées:")
        print(f"   Train: {X_train.shape}")
        print(f"   Val: {X_val.shape}")
        print(f"   Test: {X_test.shape}")

        return X_train, X_val, X_test, y_train, y_val, y_test

    except Exception as e:
        print(f"❌ Erreur chargement: {e}")
        return None, None, None, None, None, None

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
def process_all_transformations():
    """
    Fonction principale pour traiter toutes les transformations
    """
    # Chargement des données
    X_train, X_val, X_test, y_train, y_val, y_test = load_data_safely()

    if X_train is None:
        print("❌ Impossible de charger les données!")
        return

    # Initialiser le transformateur avec des paramètres optimisés
    transformer = OptimizedImageTransformer(
        image_size=64,  # Réduit pour économiser la mémoire
        n_channels_to_use=3,  # Réduit pour économiser la mémoire
        batch_size=16  # Réduit pour économiser la mémoire
    )

    print("\n🔄 Début des transformations optimisées...")

    # Liste des méthodes à traiter
    methods = ['gasf', 'mtf', 'rp']
    datasets = {
        'train': X_train,
        'val': X_val,
        'test': X_test
    }

    for method in methods:
        print(f"\n{'='*50}")
        print(f"🔄 TRANSFORMATION {method.upper()}")
        print(f"{'='*50}")

        for split_name, X_data in datasets.items():
            print(f"\n🔄 Traitement {split_name}...")

            try:
                # Transformer les données
                X_transformed = transformer.transform_with_batches(X_data, method)

                # Sauvegarder
                transformer.save_images_optimized(X_transformed, method, split_name)

                # Calculer les statistiques
                transformer.calculate_image_statistics(X_transformed, method)

                # Libérer la mémoire
                del X_transformed
                gc.collect()

                print(f"✅ {method.upper()} {split_name} terminé!")

            except Exception as e:
                print(f"❌ Erreur {method} {split_name}: {e}")
                continue

    print("\n" + "="*60)
    print("🎉 TOUTES LES TRANSFORMATIONS TERMINÉES!")
    print("="*60)

## Visualisation des Transformations

In [8]:
def visualize_sample_results():
    """
    Visualise des exemples de résultats
    """
    try:
        # Charger quelques échantillons pour visualisation
        sample_gasf = np.load(IMAGE_DATA_PATH + 'gasf/X_train_gasf.npy')[:4]
        sample_mtf = np.load(IMAGE_DATA_PATH + 'mtf/X_train_mtf.npy')[:4]
        sample_rp = np.load(IMAGE_DATA_PATH + 'rp/X_train_rp.npy')[:4]

        fig, axes = plt.subplots(3, 4, figsize=(16, 12))

        for i in range(4):
            axes[0, i].imshow(sample_gasf[i])
            axes[0, i].set_title(f'GASF {i+1}')
            axes[0, i].axis('off')

            axes[1, i].imshow(sample_mtf[i])
            axes[1, i].set_title(f'MTF {i+1}')
            axes[1, i].axis('off')

            axes[2, i].imshow(sample_rp[i])
            axes[2, i].set_title(f'RP {i+1}')
            axes[2, i].axis('off')

        plt.tight_layout()
        plt.savefig(PLOTS_PATH + 'optimized_transformations.png', dpi=150, bbox_inches='tight')
        plt.show()

        print("📊 Visualisation sauvegardée!")

    except Exception as e:
        print(f"❌ Erreur visualisation: {e}")



## Résumé et Vérification

In [9]:
def main():
    """
    Fonction principale
    """
    print("🚀 DÉMARRAGE DU PROCESSUS OPTIMISÉ")
    print("="*60)

    # Traiter toutes les transformations
    process_all_transformations()

    # Visualiser les résultats
    print("\n📊 Création des visualisations...")
    visualize_sample_results()

    # Résumé final
    print("\n" + "="*60)
    print("📋 RÉSUMÉ FINAL")
    print("="*60)

    methods = ['gasf', 'mtf', 'rp']
    splits = ['train', 'val', 'test']

    print("\n📊 Fichiers générés:")
    for method in methods:
        print(f"\n{method.upper()}:")
        for split in splits:
            filename = f'X_{split}_{method}.npy'
            filepath = IMAGE_DATA_PATH + f'{method}/' + filename
            if os.path.exists(filepath):
                data = np.load(filepath)
                print(f"   ✅ {filename}: {data.shape}")
            else:
                print(f"   ❌ {filename}: NON TROUVÉ")

    print("\n🎯 Optimisations appliquées:")
    print("   ✅ Traitement par lots pour économiser la mémoire")
    print("   ✅ Libération automatique de la mémoire")
    print("   ✅ Taille d'image réduite (64x64)")
    print("   ✅ Moins de canaux (3 au lieu de 6)")
    print("   ✅ Gestion d'erreurs robuste")

    print("\n🔄 Prochaine étape: Augmentation avec CWGAN-GP")
    print("   📂 Les images sont sauvegardées dans: " + IMAGE_DATA_PATH)

In [ ]:
if __name__ == "__main__":
    main()

🚀 DÉMARRAGE DU PROCESSUS OPTIMISÉ
📂 Chargement des données préparées...
✅ Données chargées:
   Train: (34368, 6, 1000)
   Val: (11457, 6, 1000)
   Test: (11457, 6, 1000)
🛠️ Transformateur optimisé initialisé:
   Taille image: 64x64
   Canaux utilisés: 3
   Taille batch: 16

🔄 Début des transformations optimisées...

🔄 TRANSFORMATION GASF

🔄 Traitement train...
🔄 Transformation GASF par lots...


GASF: 100%|██████████| 2148/2148 [05:50<00:00,  6.12it/s]


💾 Sauvegardé: X_train_gasf.npy - Shape: (34368, 64, 64, 3)

📊 Statistiques GASF:
   Shape: (34368, 64, 64, 3)
   Min: -1.0000
   Max: 1.0000
   Mean: -0.4772
   Std: 0.5260
✅ GASF train terminé!

🔄 Traitement val...
🔄 Transformation GASF par lots...


GASF: 100%|██████████| 717/717 [01:52<00:00,  6.40it/s]


💾 Sauvegardé: X_val_gasf.npy - Shape: (11457, 64, 64, 3)

📊 Statistiques GASF:
   Shape: (11457, 64, 64, 3)
   Min: -1.0000
   Max: 1.0000
   Mean: -0.4770
   Std: 0.5262
✅ GASF val terminé!

🔄 Traitement test...
🔄 Transformation GASF par lots...


...

RP: 100%|██████████| 2148/2148 [1:26:49<00:00,  2.43s/it]


💾 Sauvegardé: X_train_rp.npy - Shape: (34368, 64, 64, 3)

📊 Statistiques RP:
   Shape: (34368, 64, 64, 3)
   Min: 0.0000
   Max: 1.0000
   Mean: 0.1082
   Std: 0.2751
✅ RP train terminé!

🔄 Traitement val...
🔄 Transformation RP par lots...


RP: 100%|██████████| 717/717 [29:08<00:00,  2.44s/it]


💾 Sauvegardé: X_val_rp.npy - Shape: (11457, 64, 64, 3)

📊 Statistiques RP:
   Shape: (11457, 64, 64, 3)
   Min: 0.0000
   Max: 1.0000
   Mean: 0.1082
   Std: 0.2751
✅ RP val terminé!

🔄 Traitement test...
🔄 Transformation RP par lots...


RP:  44%|████▍     | 316/717 [12:43<15:30,  2.32s/it]